In [ ]:
from src import WhatsappClient

In [ ]:
client = WhatsappClient(DEBUG=True)
await client.initialize_playwright()

In [ ]:
await client.login()

In [ ]:
await client.extract_chat_details_from_side_pane()
# await client.search_pane_scroll_down()

In [ ]:
await client.fetch_latest_message()

In [ ]:
await client.on_new_message(lambda x: print(x))

In [ ]:
# await client.search_pane_scroll_down()
await client.chat_pane_scroll_up()

In [ ]:
await client.open_chat_panel("Prathwik")

In [ ]:
from tabulate import tabulate

messages = await client.extract_messages()

table_data = [
    [msg["sender"], msg["time"], msg["message"], str(msg["attachment"])]
    for msg in messages
]
headers = ["Sender", "Time Sent", "Message", "Attachment Details"]

print("Number of previous messages: ", len(messages))
print(tabulate(table_data, headers=headers, tablefmt="grid"))

In [1]:
## LEAVE THIS AS IT IS GOOD FOR REFACTORING STUFF
import asyncio
from playwright.async_api import async_playwright
import os

BASE_URL = "https://web.whatsapp.com"


async def initialize_playwright(browser_instance, user_data_dir, headless):
    # TODO: Perform browser level optimizations and other stuff
    playwright = await async_playwright().start()

    browser = await playwright[browser_instance].launch_persistent_context(
        user_data_dir, headless=headless
    )

    page_instance = await browser.new_page()

    # await page_instance.set_viewport_size({"width": 1920, "height": 1080})
    return playwright, browser, page_instance


async def login(page, user_data_dir):
    # TODO: QR code & phone number login via script

    await page.goto(BASE_URL)
    await page.bring_to_front()

    print("Waiting for WhatsApp chats to load...")
    await page.wait_for_selector(
        '//*[@id="pane-side"]/div[2]/div/div/child::div', timeout=600000
    )
    print("WhatsApp chats loaded.")

In [2]:
playwright, browser, page = await initialize_playwright("chromium", "user_data", False)
await login(page, "user_data")

Waiting for WhatsApp chats to load...
WhatsApp chats loaded.


In [3]:
## ADD TESTING STUFF HERE
async def trigger_on_notification():
    await page.evaluate(
        """
        const originalNotification = window.Notification;
        window.Notification = function(title, options) {
            console.log("New message received:", title, options);
            return new originalNotification(title, options);
        };
        """
    )

    def handle_console(msg):
        if "New message received:" in msg.text:
            print(f"Notification received: {msg.text}")
            # Add your notification handling logic here

    page.on("console", handle_console)

    print("Listening for notifications...")

    # Keep the function running
    while True:
        await asyncio.sleep(1)

In [4]:
await trigger_on_notification()

Error: Page.add_script_tag: Refused to execute inline script because it violates the following Content Security Policy directive: "script-src data: blob: 'self' 'report-sample' https://static.whatsapp.net https://*.youtube.com https://maps.googleapis.com https://maps.gstatic.com https://lens.google.com/upload https://*.google-analytics.com 'wasm-unsafe-eval'". Either the 'unsafe-inline' keyword, a hash ('sha256-DQzJoR9tZFb6VAomx3cPs/8Ooypp2qwZXpgNjgHioBM='), or a nonce ('nonce-...') is required to enable inline execution.
